# 05 — 平滑化フィルタ

## 目標
- Box / Gaussian / Median / Bilateral の4種を比較
- カスタムカーネルで `cv2.filter2D` を使う

## フィルタ比較
| フィルタ | エッジ保持 | ノイズ耐性 | 速度 |
|---------|----------|----------|------|
| Box     | ×        | 中       | 速   |
| Gaussian| △        | 中       | 速   |
| Median  | △        | S&P強い  | 中   |
| Bilateral| ○       | 中       | 遅   |


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import cv2
import numpy as np
from utils import DATA_DIR, load_image, show_nb

src = load_image(DATA_DIR / 'lena.png')

# ランダムノイズを加えた画像で比較
rng = np.random.default_rng(0)
noise = rng.integers(0, 50, src.shape, dtype=np.uint8)
noisy = cv2.add(src, noise)

In [ ]:
box       = cv2.blur(noisy, (9, 9))
gauss     = cv2.GaussianBlur(noisy, (9, 9), 0)
median    = cv2.medianBlur(noisy, 9)
bilateral = cv2.bilateralFilter(noisy, d=9, sigmaColor=75, sigmaSpace=75)

show_nb([
    ('src (clean)', src),
    ('noisy', noisy),
    ('box 9×9', box),
    ('Gaussian 9×9', gauss),
    ('median 9', median),
    ('bilateral d=9', bilateral),
], cols=3)

In [ ]:
# カスタムカーネル: シャープネス強調
kernel = np.array([[ 0, -1,  0],
                   [-1,  5, -1],
                   [ 0, -1,  0]], dtype=np.float32)
sharpened = cv2.filter2D(src, -1, kernel)

# エンボス
kernel_emboss = np.array([[-2, -1, 0],
                           [-1,  1, 1],
                           [ 0,  1, 2]], dtype=np.float32)
embossed = cv2.filter2D(src, -1, kernel_emboss)

show_nb([('src', src), ('sharpened', sharpened), ('emboss', embossed)], cols=3)